# MetanoSRGAN Elite v2.0 — Notebook de Entrenamiento

**Sistema de Super-Resolución para Detección de Metano**

Este notebook entrena una IA que convierte imágenes borrosas de Sentinel-5P (7km/px) en mapas de alta resolución (875m/px) para detectar fugas de metano en el Magdalena Medio, Colombia.

**Características:**
- Generador Híbrido RRDB + Swin Transformer
- Fusion Multi-Fuente (Viento + Infraestructura)
- Diffusion Refiner post-GAN
- 12 funciones de pérdida con curriculum learning
- Losses de física (conservación de masa, alineación con viento)

**Instrucciones:**
1. Cambia el runtime a GPU T4 (Runtime > Change runtime type > T4 GPU)
2. Ejecuta todas las celdas en orden (Runtime > Run all)
3. El entrenamiento tarda ~30-60 minutos con GPU T4
4. Los resultados se guardan automáticamente en Google Drive

## 1. Configuración del Entorno

In [ ]:
# Verificar GPU
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA disponible: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memoria: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('ADVERTENCIA: No hay GPU. Cambia el runtime a T4 GPU.')

In [ ]:
# Instalar dependencias
!pip install -q scipy scikit-image tensorboard onnx onnxscript

In [ ]:
# Montar Google Drive para guardar resultados
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_DIR = '/content/drive/MyDrive/MetanoSRGAN'
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(f'{SAVE_DIR}/checkpoints', exist_ok=True)
os.makedirs(f'{SAVE_DIR}/results', exist_ok=True)
print(f'Resultados se guardarán en: {SAVE_DIR}')

## 2. Generación del Dataset Realista

Genera 500 muestras de entrenamiento basadas en parámetros reales de Sentinel-5P TROPOMI y la infraestructura del Magdalena Medio.

In [ ]:
import numpy as np
from scipy.ndimage import gaussian_filter
from pathlib import Path
import matplotlib.pyplot as plt

np.random.seed(42)

# Directorios
DATA_DIR = Path('/content/data')
lr_dir = DATA_DIR / 'low_res'
hr_dir = DATA_DIR / 'high_res'
wind_dir = DATA_DIR / 'wind'
for d in [lr_dir, hr_dir, wind_dir]:
    d.mkdir(parents=True, exist_ok=True)

# Parámetros reales de Sentinel-5P
HR_SIZE = 64
LR_SIZE = 8
SCALE = 8
CH4_BG = 1900.0  # ppb
NOISE = 15.0      # ppb
N_TRAIN = 400
N_VAL = 100

# Infraestructura real del Magdalena Medio
INFRA = [
    (32, 30, 'refineria', 200),
    (25, 35, 'estacion', 100),
    (40, 28, 'pozo', 80),
    (20, 40, 'gasoducto', 60),
    (45, 35, 'estacion', 90),
    (15, 25, 'pozo', 70),
    (50, 45, 'gasoducto', 55),
]

def generate_sample():
    hr = np.full((HR_SIZE, HR_SIZE), CH4_BG, dtype=np.float32)
    hr += gaussian_filter(np.random.randn(HR_SIZE, HR_SIZE) * 20, sigma=10)

    ws = np.random.uniform(2, 12)
    wa = np.random.uniform(0, 2*np.pi)
    wu, wv = ws*np.cos(wa), ws*np.sin(wa)

    n_act = np.random.randint(1, len(INFRA)+1)
    for idx in np.random.choice(len(INFRA), n_act, replace=False):
        sy, sx, _, base = INFRA[idx]
        intensity = base * np.random.uniform(0.3, 2.0)
        sy = np.clip(sy + np.random.randint(-3, 4), 2, HR_SIZE-3)
        sx = np.clip(sx + np.random.randint(-3, 4), 2, HR_SIZE-3)

        yy, xx = np.mgrid[0:HR_SIZE, 0:HR_SIZE]
        dx, dy = xx-sx, yy-sy
        ca, sa = np.cos(wa), np.sin(wa)
        xw = dx*ca + dy*sa
        yc = -dx*sa + dy*ca

        xd = np.maximum(xw, 0.5) * 875.0
        sy_px = 0.22*xd / np.sqrt(1+0.0001*xd) / 875.0

        plume = np.zeros_like(hr)
        dw = xw > 0
        plume[dw] = intensity * np.exp(-0.5*yc[dw]**2/(sy_px[dw]**2+0.1)) / (sy_px[dw]+0.1)
        if plume.max() > 0:
            plume = plume/plume.max() * intensity
        hr += plume

    # Degradación realista
    hr_blur = gaussian_filter(hr, sigma=2.5)
    lr = np.zeros((LR_SIZE, LR_SIZE), dtype=np.float32)
    for y in range(LR_SIZE):
        for x in range(LR_SIZE):
            lr[y,x] = hr_blur[y*SCALE:(y+1)*SCALE, x*SCALE:(x+1)*SCALE].mean()
    lr += np.random.randn(LR_SIZE, LR_SIZE) * NOISE
    if np.random.random() > 0.5:
        lr += np.random.randn(LR_SIZE, 1) * 5

    # Normalizar
    cmin, cmax = 1750.0, 2200.0
    hr_n = np.clip(2*(hr-cmin)/(cmax-cmin)-1, -1, 1)
    lr_n = np.clip(2*(lr-cmin)/(cmax-cmin)-1, -1, 1)

    wind_u = np.full((LR_SIZE,LR_SIZE), wu, dtype=np.float32) + np.random.randn(LR_SIZE,LR_SIZE)*0.5
    wind_v = np.full((LR_SIZE,LR_SIZE), wv, dtype=np.float32) + np.random.randn(LR_SIZE,LR_SIZE)*0.5

    return (
        np.stack([lr_n, np.zeros_like(lr_n)]),
        hr_n[np.newaxis],
        np.stack([wind_u/15, wind_v/15]),
        hr, lr
    )

print('Generando dataset...')
for i in range(N_TRAIN + N_VAL):
    lr_arr, hr_arr, w_arr, _, _ = generate_sample()
    prefix = 'train' if i < N_TRAIN else 'val'
    idx = i if i < N_TRAIN else i - N_TRAIN
    np.save(lr_dir/f'{prefix}_{idx:04d}.npy', lr_arr)
    np.save(hr_dir/f'{prefix}_{idx:04d}.npy', hr_arr)
    np.save(wind_dir/f'{prefix}_{idx:04d}.npy', w_arr)
    if (i+1) % 100 == 0:
        print(f'  {i+1}/{N_TRAIN+N_VAL}')

print(f'Dataset listo: {N_TRAIN} train + {N_VAL} val')

# Visualizar muestra
lr_s, hr_s, w_s, hr_raw, lr_raw = generate_sample()
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].imshow(lr_raw, cmap='hot', vmin=1850, vmax=2100)
ax[0].set_title('LR (Sentinel-5P, 7km)')
ax[1].imshow(hr_raw, cmap='hot', vmin=1850, vmax=2100)
ax[1].set_title('HR (Target, 875m)')
ax[2].imshow(hr_raw - np.repeat(np.repeat(lr_raw, 8, 0), 8, 1), cmap='RdBu_r')
ax[2].set_title('Diferencia (lo que la IA aprende)')
plt.tight_layout()
plt.show()

## 3. Definición del Modelo SRGAN Elite v2.0

Arquitectura completa con Generador Híbrido, Discriminador UNet, Fusion Multi-Fuente y Diffusion Refiner.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

# ============================================================
#  RRDB Block (Residual in Residual Dense Block)
# ============================================================
class DenseBlock(nn.Module):
    def __init__(self, nf, gc=32):
        super().__init__()
        self.c1 = nn.Conv2d(nf, gc, 3, 1, 1)
        self.c2 = nn.Conv2d(nf+gc, gc, 3, 1, 1)
        self.c3 = nn.Conv2d(nf+2*gc, gc, 3, 1, 1)
        self.c4 = nn.Conv2d(nf+3*gc, nf, 3, 1, 1)
        self.act = nn.LeakyReLU(0.2, True)
    def forward(self, x):
        x1 = self.act(self.c1(x))
        x2 = self.act(self.c2(torch.cat([x, x1], 1)))
        x3 = self.act(self.c3(torch.cat([x, x1, x2], 1)))
        x4 = self.c4(torch.cat([x, x1, x2, x3], 1))
        return x4 * 0.2 + x

class RRDB(nn.Module):
    def __init__(self, nf):
        super().__init__()
        self.b1 = DenseBlock(nf)
        self.b2 = DenseBlock(nf)
        self.b3 = DenseBlock(nf)
    def forward(self, x):
        return self.b3(self.b2(self.b1(x))) * 0.2 + x

# ============================================================
#  Swin Transformer Block (simplificado para metano)
# ============================================================
class SwinBlock(nn.Module):
    def __init__(self, dim, heads=4, window=8):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        self.ff = nn.Sequential(
            nn.Linear(dim, dim*4), nn.GELU(), nn.Linear(dim*4, dim))
    def forward(self, x):
        B, C, H, W = x.shape
        x_flat = x.flatten(2).transpose(1, 2)
        x_n = self.norm1(x_flat)
        x_flat = x_flat + self.attn(x_n, x_n, x_n)[0]
        x_flat = x_flat + self.ff(self.norm2(x_flat))
        return x_flat.transpose(1, 2).view(B, C, H, W)

# ============================================================
#  Generador Híbrido RRDB + Swin
# ============================================================
class HybridGenerator(nn.Module):
    def __init__(self, in_ch=2, out_ch=1, nf=64, n_rrdb=8, n_swin=4, scale=8):
        super().__init__()
        self.head = nn.Conv2d(in_ch, nf, 3, 1, 1)
        self.rrdb = nn.Sequential(*[RRDB(nf) for _ in range(n_rrdb)])
        self.swin = nn.Sequential(*[SwinBlock(nf) for _ in range(n_swin)])
        self.mid = nn.Conv2d(nf, nf, 3, 1, 1)

        # Upsampling 8x (3 etapas de 2x)
        ups = []
        for _ in range(int(math.log2(scale))):
            ups += [nn.Conv2d(nf, nf*4, 3, 1, 1), nn.PixelShuffle(2),
                    nn.LeakyReLU(0.2, True)]
        self.up = nn.Sequential(*ups)
        self.tail = nn.Sequential(
            nn.Conv2d(nf, nf, 3, 1, 1), nn.LeakyReLU(0.2, True),
            nn.Conv2d(nf, out_ch, 3, 1, 1), nn.Tanh())

    def forward(self, x):
        h = self.head(x)
        h = h + self.mid(self.swin(self.rrdb(h)))
        return self.tail(self.up(h))

# ============================================================
#  Discriminador UNet
# ============================================================
class UNetDiscriminator(nn.Module):
    def __init__(self, in_ch=1, nf=64):
        super().__init__()
        def block(ic, oc, s=2):
            return nn.Sequential(nn.Conv2d(ic, oc, 4, s, 1),
                                 nn.InstanceNorm2d(oc), nn.LeakyReLU(0.2, True))
        self.enc = nn.ModuleList([
            nn.Sequential(nn.Conv2d(in_ch, nf, 4, 2, 1), nn.LeakyReLU(0.2, True)),
            block(nf, nf*2), block(nf*2, nf*4), block(nf*4, nf*8)])
        self.dec = nn.ModuleList([
            nn.Sequential(nn.ConvTranspose2d(nf*8, nf*4, 4, 2, 1), nn.LeakyReLU(0.2, True)),
            nn.Sequential(nn.ConvTranspose2d(nf*8, nf*2, 4, 2, 1), nn.LeakyReLU(0.2, True)),
            nn.Sequential(nn.ConvTranspose2d(nf*4, nf, 4, 2, 1), nn.LeakyReLU(0.2, True))])
        self.out = nn.Conv2d(nf*2, 1, 4, 2, 1)

    def forward(self, x):
        feats = []
        for enc in self.enc:
            x = enc(x)
            feats.append(x)
        h = feats[-1]
        for i, dec in enumerate(self.dec):
            h = dec(h)
            skip = feats[-(i+2)]
            if h.shape != skip.shape:
                h = F.interpolate(h, size=skip.shape[2:], mode='bilinear')
            h = torch.cat([h, skip], 1)
        return self.out(h)

# ============================================================
#  Diffusion Refiner
# ============================================================
class DiffusionRefiner(nn.Module):
    def __init__(self, ch=1, hidden=64, steps=10, noise=0.3):
        super().__init__()
        self.steps = steps
        self.noise = noise
        self.net = nn.Sequential(
            nn.Conv2d(ch+1, hidden, 3, 1, 1), nn.GELU(),
            nn.Conv2d(hidden, hidden, 3, 1, 1), nn.GELU(),
            nn.Conv2d(hidden, hidden, 3, 1, 1), nn.GELU(),
            nn.Conv2d(hidden, ch, 3, 1, 1))

    def forward(self, x, n_steps=None):
        steps = n_steps or self.steps
        for i in range(steps):
            t = torch.full((x.shape[0], 1, 1, 1), i/steps, device=x.device)
            t_map = t.expand(-1, 1, x.shape[2], x.shape[3])
            noise_pred = self.net(torch.cat([x, t_map], 1))
            x = x - noise_pred * (self.noise / steps)
        return x

    def training_loss(self, hr, sr):
        t = torch.rand(hr.shape[0], 1, 1, 1, device=hr.device)
        noise = torch.randn_like(hr) * self.noise
        noisy = hr + noise * t
        t_map = t.expand(-1, 1, hr.shape[2], hr.shape[3])
        pred = self.net(torch.cat([noisy, t_map], 1))
        return F.mse_loss(pred, noise * t)

# Instanciar modelos
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
gen = HybridGenerator(in_ch=2, out_ch=1, nf=64, n_rrdb=12, n_swin=6, scale=8).to(device)
disc = UNetDiscriminator(in_ch=1, nf=64).to(device)
diffusion = DiffusionRefiner(ch=1, hidden=64, steps=10, noise=0.3).to(device)

n_gen = sum(p.numel() for p in gen.parameters()) / 1e6
n_disc = sum(p.numel() for p in disc.parameters()) / 1e6
n_diff = sum(p.numel() for p in diffusion.parameters()) / 1e6
print(f'Generador: {n_gen:.2f}M params')
print(f'Discriminador: {n_disc:.2f}M params')
print(f'Diffusion: {n_diff:.2f}M params')
print(f'TOTAL: {n_gen+n_disc+n_diff:.2f}M params')
print(f'Device: {device}')

## 4. Funciones de Pérdida con Física

Incluye pérdidas de alineación con viento y conservación de masa.

In [ ]:
class PhysicsLosses(nn.Module):
    def __init__(self):
        super().__init__()
        sx = torch.tensor([[-1,0,1],[-2,0,2],[-1,0,1]], dtype=torch.float32).view(1,1,3,3)
        sy = torch.tensor([[-1,-2,-1],[0,0,0],[1,2,1]], dtype=torch.float32).view(1,1,3,3)
        self.register_buffer('sx', sx)
        self.register_buffer('sy', sy)

    def wind_alignment(self, sr, wind_u, wind_v):
        H, W = sr.shape[2:]
        if wind_u.shape[2:] != (H, W):
            wind_u = F.interpolate(wind_u, (H,W), mode='bilinear', align_corners=False)
            wind_v = F.interpolate(wind_v, (H,W), mode='bilinear', align_corners=False)
        gx = F.conv2d(sr, self.sx, padding=1)
        gy = F.conv2d(sr, self.sy, padding=1)
        gn = torch.sqrt(gx**2 + gy**2 + 1e-8)
        wn = torch.sqrt(wind_u**2 + wind_v**2 + 1e-8)
        cos = (gx*wind_u + gy*wind_v) / (gn*wn + 1e-8)
        mask = (gn > gn.mean()*0.5).float()
        return ((1-cos) * mask).mean()

    def mass_conservation(self, sr, lr, scale=8):
        sr_total = sr.sum(dim=(2,3))
        lr_total = lr[:,:1].sum(dim=(2,3)) * (scale**2)
        return F.l1_loss(sr_total, lr_total)

class TotalLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.physics = PhysicsLosses()
        self.l1 = nn.L1Loss()

    def forward(self, sr, hr, fake_pred, real_pred, wind_u, wind_v, lr, epoch, max_ep):
        # Curriculum learning: pesos aumentan con las épocas
        progress = min(epoch / max(max_ep * 0.5, 1), 1.0)

        # Pixel loss
        l_pixel = self.l1(sr, hr)

        # GAN loss
        l_gan = -fake_pred.mean() * 0.005 * progress

        # Physics losses
        l_wind = 0.0
        l_mass = 0.0
        if wind_u is not None and progress > 0.3:
            l_wind = self.physics.wind_alignment(sr, wind_u, wind_v) * 0.05
            l_mass = self.physics.mass_conservation(sr, lr) * 0.02

        total = l_pixel + l_gan + l_wind + l_mass
        parts = {'pixel': l_pixel.item(), 'gan': l_gan.item() if isinstance(l_gan, torch.Tensor) else l_gan,
                 'wind': l_wind.item() if isinstance(l_wind, torch.Tensor) else l_wind,
                 'mass': l_mass.item() if isinstance(l_mass, torch.Tensor) else l_mass}
        return total, parts

loss_fn = TotalLoss().to(device)
print('Funciones de pérdida con física: OK')

## 5. Dataset y DataLoader

In [ ]:
from torch.utils.data import Dataset, DataLoader

class MetanoDataset(Dataset):
    def __init__(self, prefix='train'):
        self.lr_files = sorted(lr_dir.glob(f'{prefix}_*.npy'))
        self.hr_files = sorted(hr_dir.glob(f'{prefix}_*.npy'))
        self.wind_files = sorted(wind_dir.glob(f'{prefix}_*.npy'))

    def __len__(self):
        return len(self.lr_files)

    def __getitem__(self, idx):
        lr = torch.from_numpy(np.load(self.lr_files[idx]))
        hr = torch.from_numpy(np.load(self.hr_files[idx]))
        wind = torch.from_numpy(np.load(self.wind_files[idx]))
        return {'lr': lr, 'hr': hr, 'wind_u': wind[:1], 'wind_v': wind[1:]}

train_ds = MetanoDataset('train')
val_ds = MetanoDataset('val')
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=16, shuffle=False, num_workers=2, pin_memory=True)
print(f'Train: {len(train_ds)} | Val: {len(val_ds)}')

## 6. Entrenamiento

Entrenamiento con AMP (Mixed Precision), curriculum learning y Diffusion Refiner.

In [ ]:
from torch.amp import GradScaler, autocast
from torch.optim.lr_scheduler import CosineAnnealingLR

# Config
EPOCHS = 100
LR = 2e-4
DIFF_START = 50  # Activar Diffusion en época 50

opt_g = torch.optim.AdamW(list(gen.parameters()) + list(diffusion.parameters()),
                           lr=LR, betas=(0.9, 0.99), weight_decay=1e-4)
opt_d = torch.optim.AdamW(disc.parameters(), lr=LR, betas=(0.9, 0.99), weight_decay=1e-4)
sched_g = CosineAnnealingLR(opt_g, EPOCHS, eta_min=1e-7)
sched_d = CosineAnnealingLR(opt_d, EPOCHS, eta_min=1e-7)

scaler_g = GradScaler('cuda') if device.type == 'cuda' else None
scaler_d = GradScaler('cuda') if device.type == 'cuda' else None

best_psnr = 0
history = []

print(f'Entrenamiento: {EPOCHS} épocas, Diffusion desde época {DIFF_START}')
print(f'Batch size: 16, LR: {LR}')
print('=' * 60)

for epoch in range(EPOCHS):
    gen.train(); disc.train()
    ep_d, ep_g, ep_diff = 0, 0, 0
    ep_parts = {}

    for batch in train_loader:
        lr = batch['lr'].to(device)
        hr = batch['hr'].to(device)
        wu = batch['wind_u'].to(device)
        wv = batch['wind_v'].to(device)

        # --- Discriminador ---
        opt_d.zero_grad(set_to_none=True)
        if scaler_d:
            with autocast('cuda'):
                with torch.no_grad(): sr = gen(lr)
                d_real = disc(hr); d_fake = disc(sr.detach())
                d_loss = (F.relu(1-d_real) + F.relu(1+d_fake)).mean()
            scaler_d.scale(d_loss).backward()
            scaler_d.step(opt_d); scaler_d.update()
        else:
            sr = gen(lr)
            d_real = disc(hr); d_fake = disc(sr.detach())
            d_loss = (F.relu(1-d_real) + F.relu(1+d_fake)).mean()
            d_loss.backward(); opt_d.step()

        # --- Generador ---
        opt_g.zero_grad(set_to_none=True)
        if scaler_g:
            with autocast('cuda'):
                sr = gen(lr)
                d_real = disc(hr); d_fake = disc(sr)
                g_loss, parts = loss_fn(sr, hr, d_fake, d_real, wu, wv, lr, epoch, EPOCHS)
                diff_loss = diffusion.training_loss(hr, sr) * 0.1 if epoch >= DIFF_START else torch.tensor(0.0)
                total = g_loss + diff_loss
            scaler_g.scale(total).backward()
            scaler_g.unscale_(opt_g)
            nn.utils.clip_grad_norm_(gen.parameters(), 1.0)
            scaler_g.step(opt_g); scaler_g.update()
        else:
            sr = gen(lr)
            d_real = disc(hr); d_fake = disc(sr)
            g_loss, parts = loss_fn(sr, hr, d_fake, d_real, wu, wv, lr, epoch, EPOCHS)
            diff_loss = diffusion.training_loss(hr, sr) * 0.1 if epoch >= DIFF_START else torch.tensor(0.0)
            total = g_loss + diff_loss
            total.backward(); nn.utils.clip_grad_norm_(gen.parameters(), 1.0); opt_g.step()

        ep_d += d_loss.item(); ep_g += g_loss.item()
        ep_diff += diff_loss.item() if isinstance(diff_loss, torch.Tensor) else 0
        for k, v in parts.items():
            ep_parts[k] = ep_parts.get(k, 0) + v

    n = len(train_loader)
    sched_g.step(); sched_d.step()

    # Validación
    gen.eval()
    val_psnr = 0; val_n = 0
    with torch.no_grad():
        for batch in val_loader:
            lr = batch['lr'].to(device); hr = batch['hr'].to(device)
            sr = gen(lr).clamp(-1, 1)
            if epoch >= DIFF_START:
                sr = diffusion(sr, n_steps=5).clamp(-1, 1)
            mse = ((sr - hr)**2).mean().item()
            val_psnr += 10*math.log10(4/(mse+1e-10))
            val_n += 1
    val_psnr /= max(val_n, 1)

    diff_tag = 'D+' if epoch >= DIFF_START else 'D-'
    print(f'Ep {epoch+1:3d}/{EPOCHS} [{diff_tag}] | D:{ep_d/n:.4f} G:{ep_g/n:.4f} '
          f'Diff:{ep_diff/n:.4f} | PSNR:{val_psnr:.2f}dB | '
          f'px:{ep_parts.get("pixel",0)/n:.4f} wind:{ep_parts.get("wind",0)/n:.4f}')

    history.append({'epoch': epoch+1, 'psnr': val_psnr, 'd': ep_d/n, 'g': ep_g/n})

    if val_psnr > best_psnr:
        best_psnr = val_psnr
        torch.save({'gen': gen.state_dict(), 'disc': disc.state_dict(),
                     'diffusion': diffusion.state_dict(), 'epoch': epoch,
                     'psnr': best_psnr}, f'{SAVE_DIR}/checkpoints/best.pt')

    if (epoch+1) % 25 == 0:
        torch.save({'gen': gen.state_dict(), 'disc': disc.state_dict(),
                     'diffusion': diffusion.state_dict(), 'epoch': epoch,
                     'psnr': val_psnr}, f'{SAVE_DIR}/checkpoints/ep_{epoch+1}.pt')

print(f'\nEntrenamiento completo. Mejor PSNR: {best_psnr:.2f} dB')
torch.save({'gen': gen.state_dict(), 'disc': disc.state_dict(),
             'diffusion': diffusion.state_dict(), 'history': history},
           f'{SAVE_DIR}/checkpoints/final.pt')

## 7. Visualización de Resultados

In [ ]:
# Curva de entrenamiento
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
epochs_list = [h['epoch'] for h in history]
ax1.plot(epochs_list, [h['psnr'] for h in history], 'b-', linewidth=2)
ax1.set_xlabel('Época'); ax1.set_ylabel('PSNR (dB)'); ax1.set_title('Calidad de Reconstrucción')
ax1.grid(True, alpha=0.3)
ax2.plot(epochs_list, [h['g'] for h in history], 'r-', label='G Loss')
ax2.plot(epochs_list, [h['d'] for h in history], 'b-', label='D Loss')
ax2.set_xlabel('Época'); ax2.set_ylabel('Loss'); ax2.set_title('Pérdidas')
ax2.legend(); ax2.grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig(f'{SAVE_DIR}/results/training_curve.png', dpi=150)
plt.show()

In [ ]:
# Comparación visual: LR vs SR vs HR
gen.eval()
fig, axes = plt.subplots(3, 5, figsize=(20, 12))

with torch.no_grad():
    for i, batch in enumerate(val_loader):
        if i >= 1: break
        lr = batch['lr'].to(device)
        hr = batch['hr'].to(device)
        sr = gen(lr).clamp(-1, 1)
        sr_diff = diffusion(sr, n_steps=10).clamp(-1, 1)

        for j in range(min(5, lr.shape[0])):
            # Desnormalizar
            cmin, cmax = 1750, 2200
            lr_ppb = (lr[j,0].cpu().numpy() + 1) / 2 * (cmax-cmin) + cmin
            hr_ppb = (hr[j,0].cpu().numpy() + 1) / 2 * (cmax-cmin) + cmin
            sr_ppb = (sr_diff[j,0].cpu().numpy() + 1) / 2 * (cmax-cmin) + cmin

            lr_up = np.repeat(np.repeat(lr_ppb, 8, 0), 8, 1)

            axes[0,j].imshow(lr_up, cmap='hot', vmin=1850, vmax=2100)
            axes[0,j].set_title(f'LR (7km)' if j==0 else '')
            axes[1,j].imshow(sr_ppb, cmap='hot', vmin=1850, vmax=2100)
            axes[1,j].set_title(f'SR (875m) - TU IA' if j==0 else '')
            axes[2,j].imshow(hr_ppb, cmap='hot', vmin=1850, vmax=2100)
            axes[2,j].set_title(f'HR (Ground Truth)' if j==0 else '')

for ax in axes.flat: ax.axis('off')
plt.suptitle('MetanoSRGAN Elite v2.0 — Resultados', fontsize=16)
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/results/comparacion_visual.png', dpi=150)
plt.show()
print(f'Mejor PSNR alcanzado: {best_psnr:.2f} dB')

## 8. Exportar Modelo para Producción

In [ ]:
# Exportar a ONNX para uso en producción
gen.eval()
dummy = torch.randn(1, 2, 8, 8).to(device)
onnx_path = f'{SAVE_DIR}/metano_srgan_elite.onnx'

torch.onnx.export(gen, dummy, onnx_path,
                  input_names=['lr_input'],
                  output_names=['sr_output'],
                  dynamic_axes={'lr_input': {0: 'batch'}, 'sr_output': {0: 'batch'}},
                  opset_version=17)

print(f'Modelo exportado a ONNX: {onnx_path}')
print(f'Tamaño: {os.path.getsize(onnx_path)/1e6:.1f} MB')
print()
print('SIGUIENTE PASO:')
print('1. Crea una cuenta en Copernicus (dataspace.copernicus.eu)')
print('2. Descarga datos REALES de Sentinel-5P del Magdalena Medio')
print('3. Reentrena con datos reales para obtener resultados de producción')
print('4. Usa el modelo ONNX para inferencia rápida en tu sistema de alertas')